# GateGuard — 자동 라벨링 + 검수 파이프라인

**흐름**
1. Google Drive 마운트
2. 의존성 설치 + 리포 클론
3. bbox 후보 생성 (JSON 또는 YOLO)
4. 이벤트 후보 생성 (jump / crawling / tailgating / normal)
5. 검수용 이미지·클립 저장 → `review_candidates.csv` 다운로드
6. CSV에서 `review` 컬럼 입력 후 `confirmed_labels.csv` 생성

> 이 노트북은 **학습을 실행하지 않습니다.** 검수 완료 데이터셋 생성 전용입니다.

## 0. Google Drive 마운트

Drive에 영상과 JSON을 미리 올려두세요.
```
내 드라이브/
  GateGuard/
    videos/
      video_2248321_2051363.mp4
    annotations/
      annotation_2248321.json
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/GateGuard'
print('Drive 마운트 완료:', os.listdir(DRIVE_ROOT) if os.path.exists(DRIVE_ROOT) else '폴더 없음 — 경로 확인 필요')

## 1. 의존성 설치 + 리포 클론

In [ ]:
!pip install -q ultralytics opencv-python-headless

In [ ]:
import os, sys

REPO_DIR = '/content/GateGuard'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/CHOSOOGEUN/GateGuard.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# auto_labeling 패키지를 import 가능하게 경로 추가
AI_DIR = os.path.join(REPO_DIR, 'ai')
if AI_DIR not in sys.path:
    sys.path.insert(0, AI_DIR)

print('경로 설정 완료:', AI_DIR)

## 2. 경로 설정 ← **여기만 수정하세요**

In [ ]:
from pathlib import Path

# ── 입력 경로 ─────────────────────────────────────────────────────────────────
# Drive에 올린 영상 경로
VIDEO_PATH = Path('/content/drive/MyDrive/GateGuard/videos/video_2248321_2051363.mp4')

# JSON 어노테이션 경로 (없으면 None → YOLO 자동 검출)
JSON_PATH  = Path('/content/drive/MyDrive/GateGuard/annotations/annotation_2248321.json')
# JSON_PATH = None  # JSON 없이 YOLO로 돌리려면 이 줄 사용

# ── 출력 경로 ─────────────────────────────────────────────────────────────────
# Drive에 결과 저장 (Colab 세션 종료 후에도 유지됨)
OUTPUT_DIR = Path('/content/drive/MyDrive/GateGuard/auto_labeling')

# ── 게이트 라인 Y 좌표 (원본 4K 기준) ─────────────────────────────────────────
GATE_LINE_Y = 1160   # 영상마다 다를 수 있으니 확인 후 수정

# ── YOLO 모델 ─────────────────────────────────────────────────────────────────
# Drive에 올린 모델 또는 repo의 기본 모델 사용
YOLO_MODEL = str(Path(REPO_DIR) / 'yolo11n.pt')
# YOLO_MODEL = '/content/drive/MyDrive/GateGuard/models/yolo11n.pt'  # Drive 경로

# ── 검출 간격 (YOLO 사용 시) ──────────────────────────────────────────────────
FRAME_STEP = 1   # 매 프레임 (느리면 3~5로 설정)

# 경로 확인
print('영상 존재:', VIDEO_PATH.exists(), VIDEO_PATH)
print('JSON 존재:', JSON_PATH.exists() if JSON_PATH else 'None (YOLO 모드)')
print('출력 디렉터리:', OUTPUT_DIR)

## 3. 설정 객체 생성

In [ ]:
from auto_labeling.auto_label_config import AutoLabelConfig

cfg = AutoLabelConfig(
    output_dir=OUTPUT_DIR,
    yolo_model_path=YOLO_MODEL,
    gate_line_y_original=GATE_LINE_Y,
    yolo_frame_step=FRAME_STEP,
)
cfg.make_dirs()
print('출력 디렉터리 생성 완료')
print('  이미지:', cfg.images_dir)
print('  클립:  ', cfg.clips_dir)
print('  CSV:   ', cfg.review_csv)

## 4. Step 1 — bbox 후보 생성

JSON이 있으면 JSON에서 추출, 없으면 YOLO로 검출합니다.

In [ ]:
from auto_labeling.auto_bbox_labeler import generate_bbox_candidates

json_path = JSON_PATH if (JSON_PATH and JSON_PATH.exists()) else None

bbox_candidates = generate_bbox_candidates(
    video_path=VIDEO_PATH,
    config=cfg,
    json_path=json_path,
)

print(f'bbox 후보 생성 완료: {len(bbox_candidates)}개')
if bbox_candidates:
    print('샘플:')
    import json
    print(json.dumps(bbox_candidates[0], indent=2, ensure_ascii=False))

## 5. Step 2 — 이벤트 후보 생성

jump / crawling / tailgating / normal 후보를 생성합니다.

In [ ]:
import cv2
from collections import Counter
from auto_labeling.auto_event_labeler import generate_event_candidates, event_candidates_to_rows

# 영상 메타 정보
cap = cv2.VideoCapture(str(VIDEO_PATH))
img_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(f'영상 정보: {img_h}px 높이, {fps}fps, {total_frames}프레임')

event_candidates = generate_event_candidates(
    bbox_candidates=bbox_candidates,
    config=cfg,
    video_name=VIDEO_PATH.name,
    image_height=img_h,
    fps=fps,
)

counts = Counter(ev.event_type for ev in event_candidates)
print('\n이벤트 후보 생성 완료:')
for ev_type, cnt in sorted(counts.items()):
    print(f'  {ev_type:12s}: {cnt}개')
print(f'  총계        : {len(event_candidates)}개')

## 6. Step 3 — 검수용 이미지·클립 저장 + review_candidates.csv 생성

> 영상 길이에 따라 시간이 걸릴 수 있습니다.

In [ ]:
from auto_labeling.export_review_assets import export_all

rows = event_candidates_to_rows(bbox_candidates, event_candidates)

csv_path = export_all(
    rows=rows,
    video_path=VIDEO_PATH,
    config=cfg,
)

print(f'\nreview_candidates.csv 저장 완료: {csv_path}')

## 7. 결과 미리보기 + 로컬 다운로드

In [ ]:
import pandas as pd

df = pd.read_csv(cfg.review_csv)
print(f'총 후보: {len(df)}행')
print(f'이벤트별 분포:')
print(df['event_candidate'].value_counts())
print()
df.head(10)

In [ ]:
# 로컬 PC로 직접 다운로드 (Drive 저장과 별개로)
from google.colab import files
files.download(str(cfg.review_csv))

## 8. 검수용 이미지 확인 (샘플)

저장된 이미지 중 몇 장을 노트북에서 바로 확인합니다.

In [ ]:
import glob
from IPython.display import Image, display

for event_type in ['jump', 'crawling', 'tailgating', 'normal']:
    imgs = sorted(glob.glob(str(cfg.images_dir / event_type / '*.jpg')))[:2]
    if not imgs:
        print(f'[{event_type}] 이미지 없음')
        continue
    print(f'\n── {event_type} ({len(glob.glob(str(cfg.images_dir / event_type / "*.jpg")))}개) ──')
    for p in imgs:
        display(Image(p, width=700))

---
## 9. 검수 후 — confirmed_labels.csv 생성

**순서:**
1. 위에서 다운로드한 `review_candidates.csv`를 열어서 `review` 컬럼에 입력
   - `yes` — 확정
   - `no` — 제외
   - `fix` — 수정 필요 (memo 컬럼에 내용 기록)
   - `unsure` — 보류
2. 수정한 CSV를 Drive에 덮어쓰기 업로드
3. 아래 셀 실행

In [ ]:
# (선택) 수정한 CSV를 업로드
# from google.colab import files
# uploaded = files.upload()  # review_candidates.csv 업로드
# import shutil
# shutil.copy(list(uploaded.keys())[0], str(cfg.review_csv))

In [ ]:
from auto_labeling.make_confirmed_labels import make_confirmed_labels, print_review_stats

# 검수 현황 먼저 확인
print_review_stats(cfg)

In [ ]:
# confirmed_labels.csv 생성 (review=yes 항목만, 원본 좌표 기준)
confirmed_path = make_confirmed_labels(cfg)

# 결과 확인
import pandas as pd
df_confirmed = pd.read_csv(confirmed_path)
print(f'\n확정 라벨: {len(df_confirmed)}개')
print(df_confirmed[['video_name', 'frame', 'final_label', 'final_event', 'coordinate_type']].head(10))

In [ ]:
# confirmed_labels.csv 다운로드
from google.colab import files
files.download(str(confirmed_path))

---
## 여러 영상 일괄 처리 (배치)

Drive에 영상 여러 개를 올려두고 한번에 처리할 때 사용합니다.

In [ ]:
from pathlib import Path
from auto_labeling.auto_bbox_labeler import generate_bbox_candidates
from auto_labeling.auto_event_labeler import generate_event_candidates, event_candidates_to_rows
from auto_labeling.export_review_assets import export_all, save_review_csv
import cv2

VIDEO_DIR = Path('/content/drive/MyDrive/GateGuard/videos')
JSON_DIR  = Path('/content/drive/MyDrive/GateGuard/annotations')  # 없으면 YOLO 모드

all_rows = []

for video_path in sorted(VIDEO_DIR.glob('*.mp4')):
    json_path = JSON_DIR / f'annotation_{video_path.stem.split("_")[1]}.json'
    json_path = json_path if json_path.exists() else None

    print(f'\n처리 중: {video_path.name}')
    bbox_cands = generate_bbox_candidates(video_path, cfg, json_path)

    cap = cv2.VideoCapture(str(video_path))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    cap.release()

    event_cands = generate_event_candidates(bbox_cands, cfg, video_path.name, h, fps)
    rows = event_candidates_to_rows(bbox_cands, event_cands)

    # 이미지/클립 저장
    cap2 = cv2.VideoCapture(str(video_path))
    from auto_labeling.export_review_assets import save_review_image, save_review_clip
    for row in rows:
        img_p  = save_review_image(row, cap2, cfg)
        clip_p = save_review_clip(row, cap2, cfg)
        if img_p:  row['image_path'] = str(img_p)
        if clip_p: row['clip_path']  = str(clip_p)
    cap2.release()

    all_rows.extend(rows)
    print(f'  후보 {len(rows)}개 추가')

# 전체 결과를 하나의 CSV로 저장
csv_path = save_review_csv(all_rows, cfg)
print(f'\n배치 완료: {len(all_rows)}개 후보 → {csv_path}')